In [1]:
import pandas as pd
import numpy as np

col = pd.read_csv('../data/raw/dft-road-casualty-statistics-collision-2025.csv',
                  dtype={'collision_index': str}, low_memory=False)
veh = pd.read_csv('../data/raw/dft-road-casualty-statistics-vehicle-2025.csv',
                  dtype={'collision_index': str}, low_memory=False)

col = col.rename(columns={'collision_index': 'accident_index',
                          'collision_year': 'accident_year',
                          'collision_severity': 'accident_severity'})
veh = veh.rename(columns={'collision_index': 'accident_index',
                          'collision_year': 'accident_year'})

print(col.shape, veh.shape)
col.head()

(101525, 44) (183948, 32)


,accident_index,accident_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,accident_severity,number_of_vehicles,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,202517M101925,2025,17M101925,447673.0,519279.0,-1.26420,54.56650,17,3,1,...,-1,0,1,1,2,E01012014,3,1,0.0,1.0
1,202517L205225,2025,17L205225,454611.0,518597.0,-1.15704,54.55967,17,3,2,...,-1,0,1,1,2,E01012173,3,1,0.0,1.0
2,2025070971700,2025,070971700,354476.0,382656.0,-2.68516,53.33884,7,3,2,...,-1,0,2,1,2,E01012444,3,1,0.0,1.0
3,2025070403274,2025,070403274,361793.0,393163.0,-2.57655,53.43386,7,3,2,...,-1,0,2,3,2,E01012468,3,1,0.0,1.0
4,2025071000604,2025,071000604,366083.0,389327.0,-2.51157,53.39967,7,3,7,...,-1,0,2,1,1,E01012551,3,1,0.0,1.0


In [2]:
col.info()

<class 'pandas.DataFrame'>
RangeIndex: 101525 entries, 0 to 101524
Data columns (total 44 columns):
 #   Column                                            Non-Null Count   Dtype  
---  ------                                            --------------   -----  
 0   accident_index                                    101525 non-null  str    
 1   accident_year                                     101525 non-null  int64  
 2   collision_ref_no                                  101525 non-null  str    
 3   location_easting_osgr                             101523 non-null  float64
 4   location_northing_osgr                            101523 non-null  float64
 5   longitude                                         101523 non-null  float64
 6   latitude                                          101523 non-null  float64
 7   police_force                                      101525 non-null  int64  
 8   accident_severity                                 101525 non-null  int64  
 9   number_of_vehic

In [3]:
col['accident_severity'].value_counts()

accident_severity
3    74881
2    25191
1     1453
Name: count, dtype: int64

In [4]:
(col == -1).sum().sort_values(ascending=False).head(15)

local_authority_district                            101525
junction_detail_historic                             87871
special_conditions_at_site                           87858
pedestrian_crossing_human_control_historic           87852
carriageway_hazards_historic                         87852
pedestrian_crossing_physical_facilities_historic     87851
junction_control                                     44487
second_road_number                                   25105
enhanced_severity_collision                          13677
trunk_road_flag                                       7108
junction_detail                                       6701
pedestrian_crossing                                   6527
carriageway_hazards                                   5346
road_surface_conditions                                634
second_road_class                                      171
dtype: int64

## Dataset overview

- Source: UK Department for Transport, Road Safety Data (STATS19), year 2025.
- Collision table: 101,525 rows and 44 columns.
- Vehicle table: 183,948 rows and 32 columns.
- Severity: Slight 74,881, Serious 25,191, Fatal 1,453.

In [5]:
print(list(veh.columns))

['accident_index', 'accident_year', 'collision_ref_no', 'vehicle_reference', 'vehicle_type', 'towing_and_articulation', 'vehicle_manoeuvre_historic', 'vehicle_manoeuvre', 'vehicle_direction_from', 'vehicle_direction_to', 'vehicle_location_restricted_lane_historic', 'vehicle_location_restricted_lane', 'junction_location', 'skidding_and_overturning', 'hit_object_in_carriageway', 'vehicle_leaving_carriageway', 'hit_object_off_carriageway', 'first_point_of_impact', 'vehicle_left_hand_drive', 'journey_purpose_of_driver_historic', 'journey_purpose_of_driver', 'sex_of_driver', 'age_of_driver', 'age_band_of_driver', 'engine_capacity_cc', 'propulsion_code', 'age_of_vehicle', 'generic_make_model', 'driver_imd_decile', 'lsoa_of_driver', 'escooter_flag', 'driver_distance_banding']


In [6]:
veh_summary = veh.groupby('accident_index').agg(
    n_vehicle_rows=('vehicle_reference', 'count'),
    has_motorcycle=('vehicle_type', lambda s: s.isin([2, 3, 4, 5, 23, 97]).any()),
    has_pedal_cycle=('vehicle_type', lambda s: (s == 1).any()),
    has_hgv=('vehicle_type', lambda s: s.isin([19, 20, 21]).any()),
    max_driver_age=('age_of_driver', lambda s: s[s > 0].max()),
).reset_index()

print(veh_summary.shape)
veh_summary.head()

(101525, 6)


,accident_index,n_vehicle_rows,has_motorcycle,has_pedal_cycle,has_hgv,max_driver_age
0,2025010551784,3,False,False,False,55.0
1,2025010551786,2,True,False,False,49.0
2,2025010551792,5,False,False,False,29.0
3,2025010551794,1,False,False,False,40.0
4,2025010551795,2,False,True,False,62.0


In [7]:
m = col.merge(veh_summary, on='accident_index', how='left', validate='one_to_one')

print(col.shape, '->', m.shape)
print('Accidents without vehicle record:', m['n_vehicle_rows'].isna().sum())
assert len(m) == len(col)
assert m['accident_index'].is_unique
print('Join OK')

# start the cleaning log (we fill it in the next steps)
log = []
def add_log(step, before, after, reason):
    log.append({'step': step, 'rows_before': before, 'rows_after': after,
                'rows_removed': before - after, 'reason': reason})
add_log('Join collision + vehicle summary', len(col), len(m), 'One row per accident, no duplicates')

(101525, 44) -> (101525, 49)
Accidents without vehicle record: 0
Join OK


## Vehicle summary and join

- The vehicle table has several rows per accident (183,948 rows for 101,525 accidents), so it was summarized to one row per accident (number of vehicles, motorcycle, pedal cycle and HGV flags, oldest driver age) before joining.
- Joined on accident_index with a one-to-one check: 101,525 rows before and after, no accident without a vehicle record.

In [8]:
n = len(m)
m = m.drop_duplicates('accident_index')
add_log('Drop duplicate accidents', n, len(m), 'accident_index must be unique')

n = len(m)
m = m.dropna(subset=['latitude', 'longitude'])
add_log('Drop missing coordinates', n, len(m), 'DBSCAN needs coordinates')

n = len(m)
m = m[m['latitude'].between(49.8, 60.9) & m['longitude'].between(-8.7, 1.8)].copy()
add_log('Drop coordinates outside Great Britain', n, len(m), 'Impossible locations')

print(len(m))
pd.DataFrame(log)

101523


,step,rows_before,rows_after,rows_removed,reason
0,Join collision + vehicle summary,101525,101525,0,"One row per accident, no duplicates"
1,Drop duplicate accidents,101525,101525,0,accident_index must be unique
2,Drop missing coordinates,101525,101523,2,DBSCAN needs coordinates
3,Drop coordinates outside Great Britain,101523,101523,0,Impossible locations


In [9]:
n = len(m)
m['date'] = pd.to_datetime(m['date'], dayfirst=True, errors='coerce')
m['hour'] = pd.to_datetime(m['time'], format='%H:%M', errors='coerce').dt.hour
m['month'] = m['date'].dt.month
m['weekday'] = m['date'].dt.day_name()
m = m.dropna(subset=['hour', 'month']).copy()
m['hour'] = m['hour'].astype(int)
m['month'] = m['month'].astype(int)
add_log('Create hour, month, weekday', n, len(m), 'Rows with unreadable date or time removed')

print(len(m))
m[['date', 'time', 'hour', 'month', 'weekday']].head()

101523


,date,time,hour,month,weekday
0,2025-02-04,15:00,15,2,Tuesday
1,2025-05-07,09:35,9,5,Wednesday
2,2025-11-27,18:18,18,11,Thursday
3,2025-05-15,16:30,16,5,Thursday
4,2025-12-08,07:45,7,12,Monday


In [10]:
for c in ['speed_limit', 'road_type', 'light_conditions', 'weather_conditions',
          'road_surface_conditions', 'urban_or_rural_area', 'accident_severity']:
    print(c)
    print(m[c].value_counts().sort_index().to_string())
    print()

speed_limit
speed_limit
20    20043
30    49673
40     8886
50     4612
60    12709
70     5600

road_type
road_type
1     6795
2     2398
3    14610
6    74149
7     1619
9     1952

light_conditions
light_conditions
-1       16
 1    72812
 4    20705
 5      720
 6     5682
 7     1588

weather_conditions
weather_conditions
1    83404
2    10229
3      235
4      715
5      880
6       19
7      342
8     3314
9     2385

road_surface_conditions
road_surface_conditions
-1      634
 1    75678
 2    22311
 3      146
 4     1261
 5      151
 9     1342

urban_or_rural_area
urban_or_rural_area
1    67148
2    34373
3        2

accident_severity
accident_severity
1     1453
2    25190
3    74880



In [11]:
# 1. keep only real speed limits
n = len(m)
m = m[m['speed_limit'].isin([20, 30, 40, 50, 60, 70])].copy()
add_log('Drop invalid speed limits', n, len(m), 'Speed limit missing or not a standard value')

# 2. code -> text (check these against the data guide!)
maps = {
    'accident_severity': {1: 'Fatal', 2: 'Serious', 3: 'Slight'},
    'road_type': {1: 'Roundabout', 2: 'One way street', 3: 'Dual carriageway',
                  6: 'Single carriageway', 7: 'Slip road', 12: 'One way street / Slip road'},
    'light_conditions': {1: 'Daylight', 4: 'Darkness - lights lit', 5: 'Darkness - lights unlit',
                         6: 'Darkness - no lighting', 7: 'Darkness - lighting unknown'},
    'weather_conditions': {1: 'Fine', 2: 'Raining', 3: 'Snowing', 4: 'Fine + high winds',
                           5: 'Raining + high winds', 6: 'Snowing + high winds',
                           7: 'Fog or mist', 8: 'Other'},
    'road_surface_conditions': {1: 'Dry', 2: 'Wet or damp', 3: 'Snow', 4: 'Frost or ice', 5: 'Flood'},
    'urban_or_rural_area': {1: 'Urban', 2: 'Rural', 3: 'Unallocated'},
}

for c, mp in maps.items():
    unmapped = sorted(set(m[c].unique()) - set(mp))
    print(c, '-> codes turned into Unknown:', unmapped)
    m[c] = m[c].map(mp).fillna('Unknown')

print(len(m))
m['accident_severity'].value_counts()

accident_severity -> codes turned into Unknown: []
road_type -> codes turned into Unknown: [np.int64(9)]
light_conditions -> codes turned into Unknown: [np.int64(-1)]
weather_conditions -> codes turned into Unknown: [np.int64(9)]
road_surface_conditions -> codes turned into Unknown: [np.int64(-1), np.int64(9)]
urban_or_rural_area -> codes turned into Unknown: []
101523


accident_severity
Slight     74880
Serious    25190
Fatal       1453
Name: count, dtype: int64

In [12]:
final_cols = ['accident_index', 'latitude', 'longitude', 'hour', 'weekday', 'month',
              'speed_limit', 'road_type', 'light_conditions', 'weather_conditions',
              'road_surface_conditions', 'urban_or_rural_area',
              'number_of_vehicles', 'number_of_casualties', 'accident_severity']

clean = m[final_cols].copy()

for c in ['speed_limit', 'number_of_vehicles', 'number_of_casualties']:
    clean[c] = clean[c].astype(int)

assert list(clean.columns) == final_cols
assert clean.isna().sum().sum() == 0
assert clean['accident_index'].is_unique
assert clean['hour'].between(0, 23).all() and clean['month'].between(1, 12).all()
assert set(clean['accident_severity']) <= {'Fatal', 'Serious', 'Slight'}

print(clean.shape)
print('Kept', round(100 * len(clean) / len(col), 2), 'percent of the original accidents')
clean.head()

(101523, 15)
Kept 100.0 percent of the original accidents


,accident_index,latitude,longitude,hour,weekday,month,speed_limit,road_type,light_conditions,weather_conditions,road_surface_conditions,urban_or_rural_area,number_of_vehicles,number_of_casualties,accident_severity
0,202517M101925,54.56650,-1.26420,15,Tuesday,2,30,Single carriageway,Daylight,Fine,Dry,Urban,1,1,Slight
1,202517L205225,54.55967,-1.15704,9,Wednesday,5,30,Single carriageway,Daylight,Fine,Dry,Urban,2,1,Slight
2,2025070971700,53.33884,-2.68516,18,Thursday,11,30,Single carriageway,Darkness - lights lit,Fine,Wet or damp,Rural,2,2,Slight
3,2025070403274,53.43386,-2.57655,16,Thursday,5,40,Single carriageway,Daylight,Fine,Dry,Rural,2,1,Slight
4,2025071000604,53.39967,-2.51157,7,Monday,12,70,Dual carriageway,Darkness - lights lit,Fine,Wet or damp,Rural,7,1,Slight


In [13]:
clean.to_csv('../data/clean/accidents_clean.csv', index=False, float_format='%.6f')

extra_cols = ['accident_index', 'n_vehicle_rows', 'has_motorcycle',
              'has_pedal_cycle', 'has_hgv', 'max_driver_age']
m[extra_cols].to_csv('../data/clean/accidents_vehicle_features.csv', index=False)

pd.DataFrame(log).to_csv('../data/clean/cleaning_log.csv', index=False)

# test the saved file, not the one in memory
chk = pd.read_csv('../data/clean/accidents_clean.csv', dtype={'accident_index': str})
assert list(chk.columns) == final_cols
assert chk.isna().sum().sum() == 0
assert chk['accident_index'].is_unique
assert set(chk['accident_severity']) <= {'Fatal', 'Serious', 'Slight'}
print('Saved file OK:', chk.shape)
chk.head()

Saved file OK: (101523, 15)


,accident_index,latitude,longitude,hour,weekday,month,speed_limit,road_type,light_conditions,weather_conditions,road_surface_conditions,urban_or_rural_area,number_of_vehicles,number_of_casualties,accident_severity
0,202517M101925,54.56650,-1.26420,15,Tuesday,2,30,Single carriageway,Daylight,Fine,Dry,Urban,1,1,Slight
1,202517L205225,54.55967,-1.15704,9,Wednesday,5,30,Single carriageway,Daylight,Fine,Dry,Urban,2,1,Slight
2,2025070971700,53.33884,-2.68516,18,Thursday,11,30,Single carriageway,Darkness - lights lit,Fine,Wet or damp,Rural,2,2,Slight
3,2025070403274,53.43386,-2.57655,16,Thursday,5,40,Single carriageway,Daylight,Fine,Dry,Rural,2,1,Slight
4,2025071000604,53.39967,-2.51157,7,Monday,12,70,Dual carriageway,Darkness - lights lit,Fine,Wet or damp,Rural,7,1,Slight
